# Audit and generate thumbnails from synthetic image records

**Audience:** Python developers preparing reproducible data workflows. **Time:** about 15–25 minutes. **Execution:** offline; no credentials, paid APIs, or downloaded datasets.

Validate image bytes and metadata, reject malformed or oversized inputs, produce bounded thumbnails, and reconcile a per-record audit summary.

## Prerequisites and setup

Use the tested Python environment and exact dependency versions in the [collection setup](../../../README.md). Install from the collection root before opening this notebook, then select that environment's kernel. This notebook uses Python standard library, pandas and Pillow. Restart the kernel and run all cells in order. No working-directory assumptions or private files are required.

## Data provenance

All examples below are newly authored synthetic fixtures. Names, identifiers, coordinates and values are invented for this tutorial and do not represent real people, accounts, organizations or operational records. Code is newly authored for this example; the references explain the underlying APIs.

Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Generate invented images in memory

No screenshots or real photographs are embedded. Pillow draws three simple geometric fixtures; a fourth record contains invalid bytes to exercise the failure path. Metadata is deliberately wrong for one valid image.

In [ ]:
import warnings
from collections import Counter
from io import BytesIO

import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw, UnidentifiedImageError


def make_image(size, color):
    image = Image.new("RGB", size, color)
    ImageDraw.Draw(image).rectangle(
        (10, 10, min(50, size[0] - 1), min(50, size[1] - 1)), outline="white", width=3
    )
    stream = BytesIO()
    image.save(stream, format="PNG")
    image.close()
    return stream.getvalue()


records = [
    {"id": "image-1", "bytes": make_image((320, 160), "navy"), "expected_size": (320, 160)},
    {"id": "image-2", "bytes": make_image((80, 160), "green"), "expected_size": (80, 160)},
    {"id": "image-3", "bytes": make_image((120, 120), "purple"), "expected_size": (100, 100)},
    {"id": "image-4", "bytes": b"not an image", "expected_size": (10, 10)},
]

## 2. Validate before decoding the full raster

A compressed image's byte count does not bound decoded memory. Check dimensions, promote Pillow's decompression warnings to errors, verify the encoded stream, then reopen it for conversion. This example accepts PNG and JPEG only and discards source metadata when creating a thumbnail.

In [ ]:
def audit_image(record, max_bytes=1_000_000, max_pixels=1_000_000):
    payload = record["bytes"]
    if len(payload) > max_bytes:
        return {"id": record["id"], "status": "byte_limit"}, None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", Image.DecompressionBombWarning)
            with Image.open(BytesIO(payload)) as image:
                if image.format not in {"PNG", "JPEG"}:
                    return {"id": record["id"], "status": "unsupported_format"}, None
                width, height = image.size
                if width * height > max_pixels:
                    return {"id": record["id"], "status": "pixel_limit"}, None
                image.verify()
            with Image.open(BytesIO(payload)) as image:
                image.load()
                thumb = image.convert("RGB")
                thumb.info.clear()
                thumb.thumbnail((64, 64), Image.Resampling.LANCZOS)
        status = "ok" if (width, height) == record["expected_size"] else "metadata_mismatch"
        return {
            "id": record["id"],
            "status": status,
            "width": width,
            "height": height,
            "thumbnail_width": thumb.width,
            "thumbnail_height": thumb.height,
        }, thumb
    # Pillow reports a malformed PNG chunk checksum as SyntaxError.
    except (
        UnidentifiedImageError,
        OSError,
        ValueError,
        SyntaxError,
        Image.DecompressionBombError,
        Image.DecompressionBombWarning,
    ):
        return {"id": record["id"], "status": "invalid_image"}, None


results = [audit_image(record) for record in records]
report = pd.DataFrame(row for row, _ in results)
report

## 3. Reconcile counts and inspect the output

A mismatch is reported rather than silently replacing the source metadata. The thumbnail can still be useful for manual review. The first source is 2:1; its thumbnail should retain that ratio as 64×32.

In [ ]:
counts = Counter(row["status"] for row, _ in results)
assert counts == {"ok": 2, "metadata_mismatch": 1, "invalid_image": 1}
assert sum(counts.values()) == len(records)
assert results[0][1].size == (64, 32)
assert results[1][1].size == (32, 64)
for _, thumb in results:
    if thumb is not None:
        assert max(thumb.size) <= 64
        assert not thumb.info
        display(thumb)
print(dict(counts))

## 4. Test resource limits and close image buffers

The limits are local validation controls, not a promise that arbitrary image decoders can never have vulnerabilities. Keep Pillow patched and use an isolated worker with CPU/memory limits for untrusted bulk ingestion.

A valid PNG header is not sufficient: damaged image-data checksums must become an `invalid_image` record without aborting the remaining audit. The next check changes one checksum byte in our authored fixture, while retaining its dimensions and encoded content.

In [ ]:
assert audit_image(records[0], max_bytes=1)[0]["status"] == "byte_limit"
assert audit_image(records[0], max_pixels=100)[0]["status"] == "pixel_limit"
# Corrupt an IDAT checksum without changing the image header or dimensions.
corrupt_png = bytearray(records[0]["bytes"])
position = 8  # PNG signature length; each chunk has length, type, data and CRC.
while position < len(corrupt_png):
    data_length = int.from_bytes(corrupt_png[position : position + 4], "big")
    if corrupt_png[position + 4 : position + 8] == b"IDAT":
        corrupt_png[position + 8 + data_length] ^= 1
        break
    position += 12 + data_length
else:
    raise AssertionError("Synthetic PNG has no image-data chunk")
corrupt_record = {**records[0], "id": "corrupt-checksum", "bytes": bytes(corrupt_png)}
checksum_result, checksum_thumbnail = audit_image(corrupt_record)
assert checksum_result["status"] == "invalid_image"
assert checksum_thumbnail is None
# A following valid record is still processed after the corrupt one.
followup_result, followup_thumbnail = audit_image(records[1])
assert followup_result["status"] == "ok"
followup_thumbnail.close()
for _, thumb in results:
    if thumb is not None:
        thumb.close()
print("Image integrity, size, metadata and thumbnail checks passed.")

## Validation and interpretation

The assertions above are executable acceptance criteria. A failed assertion is a failed example: inspect the inputs and kernel versions before continuing. Inspect the displayed tables or figures as well; successful execution alone does not establish fitness for a different dataset.

## Limitations and extensions

The workflow audits in-memory fixtures rather than a live document store. It does not test image meaning, accessibility descriptions, orientation metadata, transparency or animated formats. Add these requirements explicitly before processing production media. A public notebook should never retain customer screenshots in output cells.

## Cleanup

This notebook creates no persistent files, network sessions or cloud resources. Restart the kernel to discard in-memory objects. If you adapt it to real data, keep inputs outside a public repository and review every output before sharing.

## References and attribution

- [Pillow Image verification, thumbnail and decompression safeguards](https://pillow.readthedocs.io/en/stable/reference/Image.html)

The example is prepared for editorial review; it is not an IBM endorsement or a claim of production certification.